# Ensemble Learning Study: Bagging vs Boosting vs Stacking
**Dataset:** Cardiac Patient Bed Head Ticket dataset (Teaching Hospital Karapitiya, Galle, Sri Lanka; Rajapaksha et al., *BioMedInformatics* 2024; Zenodo 7603772, CC BY 4.0)

**Task:** predict in-hospital death of cardiac ward patients from the first 6 hourly readings plus admission labs/history.

**Design decisions**
- One row per patient (112 rows) so that hourly rows from the same patient never end up in both train and test.
- Early window = first 6 readings per patient, because most deaths happen early.
- Positive class = **death** (19 patients), so recall means "deaths caught".
- All imputation / scaling happens inside the CV pipeline (no leakage). No synthetic data is used for evaluation.
- Evaluation = repeated stratified 5-fold CV (5 repeats), reporting AUC, PR-AUC, recall, precision, F1, balanced accuracy and Brier score.

**Sections:** 1 Setup, 2 Load and audit, 3 Cleaning, 4 Patient-level features, 5 Models, 6 Cross-validated evaluation, 7 Statistical comparison, 8 Feature importance, 9 Save results.

## 1. Setup

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier)
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
SEED = 42
WINDOW = 6
N_SPLITS, N_REPEATS = 5, 5
OUT_DIR = "notebook_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


CANDIDATES = ["../data/CardiacPatientData.csv"]
DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))
print("Using", DATA_PATH)

## 2. Load and audit the raw data

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("rows:", raw.shape[0], "| columns:", raw.shape[1], "| patients:", raw.ID.nunique())
display(raw.head())
display(raw.describe().T[["min","50%","max"]])

per_pat = raw.groupby("ID").agg(n=("ID","size"), outcome=("Outcome","first"))
print("\nOutcome is constant within patient:", (raw.groupby("ID").Outcome.nunique()==1).all())
print("Patients: survived =", (per_pat.outcome==1).sum(), "| died =", (per_pat.outcome==0).sum())
print("Median readings: survived =", per_pat[per_pat.outcome==1].n.median(),
      "| died =", per_pat[per_pat.outcome==0].n.median(), "  (stay length leaks the outcome -> never use it as a feature)")
print("Dead patients with >= 6 / 12 / 24 readings:",
      [(per_pat[(per_pat.outcome==0)].n>=h).sum() for h in (6,12,24)], "of 19")
print("\nMissing values per column (row level):")
print(raw.isna().sum()[lambda s: s>0])

## 3. Cleaning: remove physiologically impossible values
Values outside plausible limits are set to NaN **before** aggregation (e.g. GCS > 15, DBP = 999). Body temperature is stored in whole degrees Fahrenheit.

In [ ]:
LIMITS = {"SBP":(40,300), "DBP":(20,200), "HR":(20,250), "RR":(4,60),
          "BT":(90,108), "SpO2":(50,100), "GCS":(3,15)}
df = raw.sort_values("ID", kind="stable").copy()    # row order inside a patient = time order
removed = {}
for col,(lo,hi) in LIMITS.items():
    m = (df[col]<lo) | (df[col]>hi)
    removed[col] = int(m.sum()); df.loc[m, col] = np.nan
print("Impossible values set to NaN:", removed)

## 4. Patient-level feature table
For each vital: mean, min, max, last value and linear slope over the first 6 readings. Labs, demographics and history use the patient's first recorded (admission) value. Every patient has at least one recorded value for every lab/history column, so no patient-level imputation is needed for these.

In [ ]:
VITALS = ["SBP","DBP","HR","RR","BT","SpO2","GCS"]
STATIC = ["Age","Gender","Na","K","Cl","Urea","Ceratinine","Alcoholic","Smoke","FHCD","TriageScore"]

def build_patient_table(d, window=WINDOW):
    rows = []
    for pid, g in d.groupby("ID", sort=True):
        w = g.head(window)
        r = {"ID": pid, "Death": int(g.Outcome.iloc[0] == 0)}     # 1 = died (positive class)
        for v in VITALS:
            s = w[v].dropna()
            r[f"{v}_mean"]  = s.mean()      if len(s)      else np.nan
            r[f"{v}_min"]   = s.min()       if len(s)      else np.nan
            r[f"{v}_max"]   = s.max()       if len(s)      else np.nan
            r[f"{v}_last"]  = s.iloc[-1]    if len(s)      else np.nan
            r[f"{v}_slope"] = np.polyfit(np.arange(len(s)), s.values, 1)[0] if len(s) >= 3 else np.nan
        for c in STATIC:
            s = g[c].dropna(); r[c] = s.iloc[0] if len(s) else np.nan
        r["n_window"] = len(w)
        rows.append(r)
    return pd.DataFrame(rows)

pat = build_patient_table(df)
NON_FEATURES = ["ID", "Death", "n_window"]
FEATURES = [c for c in pat.columns if c not in NON_FEATURES]
X, y = pat[FEATURES], pat["Death"].values
print("patients:", len(pat), "| features:", len(FEATURES), "| deaths:", y.sum(), f"({y.mean():.1%})")
print("window length counts:", pat.n_window.value_counts().sort_index().to_dict())
print("Columns with missing values:", X.isna().sum()[lambda s: s>0].to_dict())
pat.to_csv(f"{OUT_DIR}/cardiac_patient_level.csv", index=False)

## 5. Models
Class imbalance (about 1 death per 5 survivors) is handled with class weights, not synthetic oversampling. Every model sits in a pipeline with median imputation fitted inside each fold.

| Family | Models |
|---|---|
| Baselines | Logistic regression, single decision tree |
| Bagging | Bagged trees, **Random Forest**, Extra Trees |
| Boosting | AdaBoost, sklearn Gradient Boosting, **XGBoost**, LightGBM, CatBoost |
| Stacking | RF + XGBoost + LightGBM + LR meta-learner |
| Hybrid (section 6b) | Random Forest + ANN blend, plus the ANN alone as a reference |

Hyperparameters are deliberately modest and fixed (112 rows cannot support heavy tuning without overfitting the CV). A tuning step with nested CV can come later.

In [ ]:
NEG, POS = (y==0).sum(), (y==1).sum()
SPW = round(NEG / POS, 2)

def pipe(model, scale=False):
    steps = [("imp", SimpleImputer(strategy="median"))]
    if scale: steps.append(("sc", StandardScaler()))
    steps.append(("clf", model))
    return Pipeline(steps)

rf  = lambda: RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced_subsample",
                                     n_jobs=1, random_state=SEED)
xgb = lambda: XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                            min_child_weight=1, scale_pos_weight=SPW, eval_metric="logloss",
                            n_jobs=1, random_state=SEED, verbosity=0)
lgb = lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=8, min_child_samples=5,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.8, class_weight="balanced",
                             n_jobs=1, random_state=SEED, verbose=-1)
cat = lambda: CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05, auto_class_weights="Balanced",
                                 random_seed=SEED, verbose=0, thread_count=1, allow_writing_files=False)

MODELS = {
    "Logistic Regression": pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), scale=True),
    "Decision Tree":       pipe(DecisionTreeClassifier(max_depth=4, min_samples_leaf=3, class_weight="balanced", random_state=SEED)),
    "Bagging (trees)":     pipe(BaggingClassifier(DecisionTreeClassifier(min_samples_leaf=2, class_weight="balanced"),
                                                  n_estimators=300, random_state=SEED, n_jobs=1)),
    "Random Forest":       pipe(rf()),
    "Extra Trees":         pipe(ExtraTreesClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced_subsample",
                                                     n_jobs=1, random_state=SEED)),
    "AdaBoost":            pipe(AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=SEED)),
    "Gradient Boosting":   pipe(GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                                           subsample=0.8, random_state=SEED)),
    "XGBoost":             pipe(xgb()),
    "LightGBM":            pipe(lgb()),
    "CatBoost":            pipe(cat()),
    "Stacking (RF+XGB+LGBM -> LR)": pipe(StackingClassifier(
        estimators=[("rf", rf()), ("xgb", xgb()), ("lgb", lgb())],
        final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
        stack_method="predict_proba", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=1)),
}
print(len(MODELS), "models;", "scale_pos_weight =", SPW)

## 6. Cross-validated evaluation
Repeated stratified 5-fold CV (25 train/test splits). With only 19 deaths, each test fold holds about 4 deaths, so single-fold numbers are very noisy: look at the **mean and spread across all 25 folds**. Recall, precision and F1 use the default 0.5 threshold; AUC and PR-AUC are threshold-free.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
SCORING = {
    "AUC":       "roc_auc",
    "PR_AUC":    "average_precision",
    "Recall":    make_scorer(recall_score, zero_division=0),
    "Precision": make_scorer(precision_score, zero_division=0),
    "F1":        make_scorer(f1_score, zero_division=0),
    "BalAcc":    "balanced_accuracy",
    "Brier":     "neg_brier_score",
}
fold_scores = {}
for name, model in MODELS.items():
    r = cross_validate(model, X, y, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    fold_scores[name] = {k: r[f"test_{k}"] * (-1 if k == "Brier" else 1) for k in SCORING}
    print(f"{name:32s} AUC {fold_scores[name]['AUC'].mean():.3f}  Recall {fold_scores[name]['Recall'].mean():.3f}")

### 6b. Hybrid: Random Forest + ANN blend
The hybrid averages a Random Forest and a small neural network: `p = w * p_RF + (1 - w) * p_ANN`. The weight `w` is chosen from {0.25, 0.5, 0.75} using out-of-fold predictions made *inside each training fold* (inner 3-fold CV), so the test fold never influences it.

The network is a small scikit-learn MLP (two hidden layers of 16 and 8 units, L2 penalty), averaged over 5 random initialisations. Deaths are randomly oversampled to parity **inside the training data only**, because the MLP cannot take class weights. The ANN alone is evaluated too, so the hybrid can be compared with both of its parts.

The code lives in `src/models.py` (`OversampledMLP`, `HybridRFANN`) and is imported here so the notebook and the dashboard use identical models.

In [ ]:
import sys
for root in ("..", "."):
    if os.path.exists(os.path.join(root, "src", "models.py")) and os.path.abspath(root) not in sys.path:
        sys.path.insert(0, os.path.abspath(root))
from src.models import HybridRFANN, OversampledMLP

HYBRIDS = {
    "Hybrid (RF + ANN blend)": pipe(HybridRFANN()),
    "ANN (MLP)":               pipe(OversampledMLP()),
}
MODELS.update(HYBRIDS)
for name, model in HYBRIDS.items():
    r = cross_validate(model, X, y, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    fold_scores[name] = {k: r[f"test_{k}"] * (-1 if k == "Brier" else 1) for k in SCORING}
    print(f"{name:32s} AUC {fold_scores[name]['AUC'].mean():.3f}  Recall {fold_scores[name]['Recall'].mean():.3f}")

In [ ]:
rows = []
for name, s in fold_scores.items():
    row = {"Model": name}
    for k, v in s.items():
        row[k] = f"{v.mean():.3f} ± {v.std():.3f}"
    row["_auc"] = s["AUC"].mean()
    rows.append(row)
summary = pd.DataFrame(rows).sort_values("_auc", ascending=False).drop(columns="_auc").reset_index(drop=True)
display(summary)

In [ ]:
order = sorted(fold_scores, key=lambda n: fold_scores[n]["AUC"].mean())
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metric in zip(axes, ["AUC", "Recall"]):
    ax.boxplot([fold_scores[n][metric] for n in order], vert=False)
    ax.set_yticks(range(1, len(order)+1)); ax.set_yticklabels(order)
    ax.set_title(f"{metric} across {N_SPLITS*N_REPEATS} CV folds"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/cv_boxplots.png", dpi=150); plt.show()

## 7. Is Random Forest really different from the boosters?
Fold scores are correlated (training sets overlap), so an ordinary paired t-test is over-confident. We use the **Nadeau–Bengio corrected resampled t-test**. With 112 patients and 19 events, expect most differences to be *not* statistically significant, and that is itself a finding worth reporting.

In [ ]:
def corrected_ttest(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b); k = len(d)
    var = d.var(ddof=1)
    if var == 0: return 0.0, 1.0
    t = d.mean() / np.sqrt((1/k + n_test/n_train) * var)
    return t, 2 * stats.t.sf(abs(t), df=k-1)

n_test  = len(y) // N_SPLITS
n_train = len(y) - n_test
ref = "Random Forest"
out = []
for other in fold_scores:
    if other == ref: continue
    for m in ["AUC", "PR_AUC"]:
        t, p = corrected_ttest(fold_scores[ref][m], fold_scores[other][m], n_train, n_test)
        out.append({"Compared with RF": other, "Metric": m,
                    "Mean diff (RF - other)": round(fold_scores[ref][m].mean() - fold_scores[other][m].mean(), 4),
                    "t": round(t, 2), "p (corrected)": round(p, 3)})
display(pd.DataFrame(out).pivot(index="Compared with RF", columns="Metric",
        values=["Mean diff (RF - other)", "p (corrected)"]))

In [ ]:
hyb = "Hybrid (RF + ANN blend)"
rows = []
for other in ["Random Forest", "ANN (MLP)", "XGBoost", "Stacking (RF+XGB+LGBM -> LR)"]:
    for m in ["AUC", "PR_AUC"]:
        t, p = corrected_ttest(fold_scores[hyb][m], fold_scores[other][m], n_train, n_test)
        rows.append({"Hybrid vs": other, "Metric": m,
                     "Mean diff (hybrid - other)": round(fold_scores[hyb][m].mean() - fold_scores[other][m].mean(), 4),
                     "p (corrected)": round(p, 3)})
display(pd.DataFrame(rows).pivot(index="Hybrid vs", columns="Metric", values=["Mean diff (hybrid - other)", "p (corrected)"]))

## 8. Feature importance (RF vs XGBoost)
Fitted on all 112 patients purely for **interpretation** (not for performance claims). Agreement between a bagging model and a boosting model on the top features is more convincing than either alone.

In [ ]:
rf_m, xgb_m = MODELS["Random Forest"], MODELS["XGBoost"]
rf_m.fit(X, y); xgb_m.fit(X, y)
imp = pd.DataFrame({"RF": rf_m.named_steps["clf"].feature_importances_,
                    "XGBoost": xgb_m.named_steps["clf"].feature_importances_}, index=FEATURES)
top = imp.assign(avg=imp.mean(axis=1)).sort_values("avg", ascending=False).head(12).drop(columns="avg")
ax = top.iloc[::-1].plot.barh(figsize=(8, 6)); ax.set_title("Top features: RF vs XGBoost")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/feature_importance.png", dpi=150); plt.show()

## 9. Save results

In [ ]:
summary.to_csv(f"{OUT_DIR}/model_comparison_summary.csv", index=False)
pd.concat({n: pd.DataFrame(s) for n, s in fold_scores.items()}, names=["Model", "fold"]).to_csv(f"{OUT_DIR}/fold_level_scores.csv")
print("Saved to", OUT_DIR, ":", os.listdir(OUT_DIR))

## Caveats to state in the report
- **Tiny sample:** 112 patients and 19 deaths, from one unit of one hospital. Results will not generalise without external validation.
- **Metric noise:** about 4 deaths per test fold, so report mean ± std over folds and avoid ranking models on small differences.
- **Selection of the 6-reading window** was based on how early deaths occur; a sensitivity check with 3 or 12 readings is a sensible robustness test.
- **Hybrid result:** the RF + ANN blend is compared with both of its parts in section 7; with 19 deaths, expect differences to be within noise.